In [3]:
import cv2

In [4]:
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')

In [5]:
cap = cv2.VideoCapture(0) 
faces_data = []
i = 0

if not cap.isOpened():
    print("Error: Could not open webcam.")
    exit()

In [6]:
while True:
        ret, frame = cap.read()
        if not ret:
            print("Error: Could not read frame.")
            break
        
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        
        faces = face_cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5, minSize=(30, 30))
       
        for (x, y, w, h) in faces:
            cv2.rectangle(frame, (x, y), (x + w, y + h), (0, 255, 0), 2)
        
        cv2.imshow('Face Detection', frame)
        
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break
cap.release()
cv2.destroyAllWindows()

In [1]:

import tensorflow as tf

print("TensorFlow version:", tf.__version__)
print("GPUs detected:", tf.config.list_physical_devices("GPU"))


TensorFlow version: 2.21.0
GPUs detected: []


In [2]:
import torch
torch.cuda.is_available()

True

In [ ]:
import os
import cv2
import time
import requests
from deepface import DeepFace

#  Ollama function FIRST
def ask_ollama(name, emotion):
    prompt = f"""
    The camera sees a person named {name}.
    Their current emotion is {emotion}.
    
    Say something friendly in 1 short sentence.
    """

    response = requests.post(
        "http://localhost:11434/api/generate",
        json={
            "model": "llama3.2",
            "prompt": prompt,
            "stream": False
        }
    )

    return response.json()["response"]


#  Load known faces
folder = "data"
known_images = []
known_names = []

for file in os.listdir(folder):
    if file.endswith((".jpg", ".png")):
        known_images.append(os.path.join(folder, file))
        known_names.append(os.path.splitext(file)[0])

#  Fast face detector
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_frontalface_default.xml")

cap = cv2.VideoCapture(0)

frame_count = 0
name = "Unknown"
emotion = ""
reply = ""

last_chat_time = 0
chat_delay = 5

last_name = ""
last_emotion = ""

while True:
    ret, frame = cap.read()
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    faces = face_cascade.detectMultiScale(gray, 1.3, 5)
    frame_count += 1

    for (x, y, w, h) in faces:
        face = frame[y:y+h, x:x+w]

        #  Run heavy AI every 10 frames
        if frame_count % 10 == 0:
            # Emotion
            analysis = DeepFace.analyze(face, actions=['emotion'], enforce_detection=False)
            emotion = analysis[0]['dominant_emotion']

            # Recognition
            name = "Unknown"
            for img_path, person_name in zip(known_images, known_names):
                result = DeepFace.verify(img_path, face, enforce_detection=False)
                if result["verified"]:
                    name = person_name
                    break

        #  Run chatbot only when needed
        if (name != last_name or emotion != last_emotion) and (time.time() - last_chat_time > chat_delay):
            reply = ask_ollama(name, emotion)
            last_chat_time = time.time()
            last_name = name
            last_emotion = emotion

        #  Draw
        cv2.rectangle(frame, (x, y), (x+w, y+h), (0,255,0), 2)

        cv2.putText(frame, f"{name}", (x, y-30),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0,255,0), 2)

        cv2.putText(frame, f"{emotion}", (x, y-5),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255,255,0), 2)

        cv2.putText(frame, reply, (x, y+h+20),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,255,255), 2)

    cv2.imshow("Face + Emotion + Ollama", frame)

    key = cv2.waitKey(1) & 0xFF
    if key == 27 or key == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

: 

In [5]:
import torch

print("CUDA Available:", torch.cuda.is_available())
print("GPU Count:", torch.cuda.device_count())

if torch.cuda.is_available():
    print("GPU Name:", torch.cuda.get_device_name(0))

CUDA Available: True
GPU Count: 1
GPU Name: NVIDIA RTX 2000 Ada Generation Laptop GPU


In [3]:
import os
import cv2
import time
import queue
import threading
import requests
import numpy as np

from deepface import DeepFace
from sklearn.metrics.pairwise import cosine_similarity
from ultralytics import YOLO

# ==========================================
# OLLAMA
# ==========================================

def ask_ollama(name, emotion):

    prompt = f"""
    The person is {name}.
    Current emotion is {emotion}.

    Respond like a friendly emotional assistant.

    Rules:
    - Maximum 15 words
    - Mention their name if known
    - Match their emotion
    """

    try:
        response = requests.post(
            "http://localhost:11434/api/generate",
            json={
                "model": "llama3.2",
                "prompt": prompt,
                "stream": False
            },
            timeout=20
        )

        return response.json()["response"]

    except Exception as e:
        return f"Error: {e}"


# ==========================================
# LOAD KNOWN FACES (EMBEDDINGS)
# ==========================================

print("Loading known faces...")

known_embeddings = []
known_names = []

DATASET = "data"

for file in os.listdir(DATASET):

    if file.endswith((".jpg", ".png", ".jpeg")):

        img_path = os.path.join(DATASET, file)

        try:
            embedding = DeepFace.represent(
                img_path=img_path,
                model_name="ArcFace",
                detector_backend="skip",
                enforce_detection=True
            )

            known_embeddings.append(
                np.array(embedding[0]["embedding"])
            )

            known_names.append(
                os.path.splitext(file)[0]
            )

            print("Loaded:", file)

        except Exception as e:
            print(file, e)

print("Faces loaded:", len(known_names))


# ==========================================
# FACE MATCHING
# ==========================================

def recognize_face(face_img):

    try:

        result = DeepFace.represent(
    img_path=face_img,
    model_name="ArcFace",
    detector_backend="skip",
    enforce_detection=True
)

        current_embedding = np.array(
            result[0]["embedding"]
        ).reshape(1, -1)

        best_score = 0
        best_name = "Unknown"

        for emb, person in zip(
            known_embeddings,
            known_names
        ):

            sim = cosine_similarity(
                current_embedding,
                emb.reshape(1, -1)
            )[0][0]
            
            print(person, sim)
            if sim > best_score:
                best_score = sim
                best_name = person

        if best_score > 0.50:
            return best_name

        return "Unknown"

    except:
        return "Unknown"


# ==========================================
# BACKGROUND OLLAMA THREAD
# ==========================================

reply_queue = queue.Queue()

latest_reply = "Hello!"

def chatbot_worker():

    global latest_reply

    while True:

        item = reply_queue.get()

        if item is None:
            break

        name, emotion = item

        reply = ask_ollama(name, emotion)

        latest_reply = reply

        reply_queue.task_done()


threading.Thread(
    target=chatbot_worker,
    daemon=True
).start()


# ==========================================
# CAMERA
# ==========================================

cap = cv2.VideoCapture(0)

face_cascade = cv2.CascadeClassifier(
    cv2.data.haarcascades +
    "haarcascade_frontalface_default.xml"
)

frame_counter = 0

current_name = "Unknown"
current_emotion = ""

last_name = ""
last_emotion = ""

last_chat_time = 0
chat_delay = 5

# ==========================================
# MAIN LOOP
# ==========================================

while True:

    ret, frame = cap.read()

    if not ret:
        break

    frame_counter += 1

    gray = cv2.cvtColor(
        frame,
        cv2.COLOR_BGR2GRAY
    )

    faces = face_cascade.detectMultiScale(
        gray,
        scaleFactor=1.3,
        minNeighbors=5
    )

    if len(faces) > 0:

        # pick the single largest (most prominent/confident) face
        x, y, w, h = max(faces, key=lambda f: f[2] * f[3])

        face = frame[y:y+h, x:x+w]

        # ----------------------------------
        # Heavy AI every 15 frames
        # ----------------------------------

        if frame_counter % 15 == 0:

            try:

                analysis = DeepFace.analyze(
                    face,
                    actions=["emotion"],
                    enforce_detection=False
                )

                current_emotion = analysis[0][
                    "dominant_emotion"
                ]

            except:
                current_emotion = ""

            current_name = recognize_face(face)

        # ----------------------------------
        # Trigger chatbot only when changed
        # ----------------------------------

        changed = (
            current_name != last_name or
            current_emotion != last_emotion
        )

        if changed:

            if time.time() - last_chat_time > chat_delay:

                reply_queue.put(
                    (
                        current_name,
                        current_emotion
                    )
                )

                last_name = current_name
                last_emotion = current_emotion

                last_chat_time = time.time()

        # ----------------------------------
        # Drawing
        # ----------------------------------

        cv2.rectangle(
            frame,
            (x, y),
            (x+w, y+h),
            (0, 255, 0),
            2
        )

        cv2.putText(
            frame,
            f"Name: {current_name}",
            (x, y-40),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.7,
            (0,255,0),
            2
        )

        cv2.putText(
            frame,
            f"Emotion: {current_emotion}",
            (x, y-15),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.7,
            (255,255,0),
            2
        )

        cv2.putText(
            frame,
            latest_reply,
            (x, y+h+25),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.6,
            (0,255,255),
            2
        )

    cv2.imshow(
        "Emotion Chatbot v2",
        frame
    )

    key = cv2.waitKey(1)

    if key == ord("q") or key == 27:
        break

# ==========================================
# CLEANUP
# ==========================================

reply_queue.put(None)

cap.release()

cv2.destroyAllWindows()

Loading known faces...
Loaded: DongAn.jpg
Loaded: Harsh.jpg
Loaded: Sandra.jpg
Faces loaded: 3
DongAn 0.06144550215313783
Harsh 0.11843230253717996
Sandra 0.10176642905747418
DongAn 0.0882235219403547
Harsh 0.13634344088865596
Sandra 0.12178394148227231
DongAn 0.0833374316783815
Harsh 0.1268037212462083
Sandra 0.10650742277822128
DongAn 0.07473725513069605
Harsh 0.12043203665348615
Sandra 0.09951276221607827
DongAn 0.05211528891866332
Harsh 0.08906713485168294
Sandra 0.08465188240486173
DongAn 0.08143553094731981
Harsh 0.13233714635238364
Sandra 0.12104226392813154
DongAn 0.07485544995814539
Harsh 0.10777252889730699
Sandra 0.05642726912992656
DongAn 0.052549582490333945
Harsh 0.08138151580396243
Sandra 0.025825279933843526
DongAn 0.08076967934885242
Harsh 0.10860964143530863
Sandra 0.062454289039235736
DongAn 0.06196336239257903
Harsh 0.09353505134642509
Sandra 0.04939769146363219
DongAn 0.09758204599038361
Harsh 0.1280006365839587
Sandra 0.08857067581427885
DongAn 0.07337229439523374

In [2]:
import os
import cv2
import time
import queue
import threading
import requests
import numpy as np

from deepface import DeepFace
from sklearn.metrics.pairwise import cosine_similarity

# ==========================================
# OLLAMA
# ==========================================

def ask_ollama(name, emotion):

    prompt = f"""
    The person is {name}.
    Current emotion is {emotion}.

    Respond like a friendly emotional assistant.

    Rules:
    - Maximum 15 words
    - Mention their name if known
    - Match their emotion
    """

    try:
        response = requests.post(
            "http://localhost:11434/api/generate",
            json={
                "model": "llama3.2",
                "prompt": prompt,
                "stream": False
            },
            timeout=20
        )

        return response.json()["response"]

    except Exception as e:
        return f"Error: {e}"


# ==========================================
# LOAD KNOWN FACES (EMBEDDINGS)
# ==========================================

print("Loading known faces...")

known_embeddings = []
known_names = []

DATASET = "backend/data"

for file in os.listdir(DATASET):

    if file.endswith((".jpg", ".png", ".jpeg")):

        img_path = os.path.join(DATASET, file)

        try:
            embedding = DeepFace.represent(
                img_path=img_path,
                model_name="ArcFace",
                detector_backend="retinaface",
                enforce_detection=True,
                align=True
            )

            known_embeddings.append(
                np.array(embedding[0]["embedding"])
            )

            known_names.append(
                os.path.splitext(file)[0]
            )

            print("Loaded:", file)

        except Exception as e:
            print(file, e)

print("Faces loaded:", len(known_names))

# Stack into a matrix once for fast vectorized comparison
if known_embeddings:
    known_matrix = np.vstack(known_embeddings)  # shape (N, D)
else:
    known_matrix = np.empty((0, 512))

RECOGNITION_THRESHOLD = 0.68


# ==========================================
# FACE MATCHING (operates on padded region of the FULL frame,
# not a tight Haar crop, so DeepFace can detect + align properly)
# ==========================================

def recognize_face(frame, box):

    x, y, w, h = box

    pad_x, pad_y = int(w * 0.4), int(h * 0.4)
    x1 = max(0, x - pad_x)
    y1 = max(0, y - pad_y)
    x2 = min(frame.shape[1], x + w + pad_x)
    y2 = min(frame.shape[0], y + h + pad_y)

    padded_face = frame[y1:y2, x1:x2]

    try:
        result = DeepFace.represent(
            img_path=padded_face,
            model_name="ArcFace",
            detector_backend="retinaface",
            enforce_detection=True,
            align=True
        )

        current_embedding = np.array(
            result[0]["embedding"]
        ).reshape(1, -1)

        if known_matrix.shape[0] == 0:
            return "Unknown"

        sims = cosine_similarity(current_embedding, known_matrix)[0]
        best_idx = int(np.argmax(sims))
        best_score = sims[best_idx]

        if best_score > RECOGNITION_THRESHOLD:
            return known_names[best_idx]

        return "Unknown"

    except Exception:
        return "Unknown"


def analyze_emotion(padded_face):

    try:
        analysis = DeepFace.analyze(
            padded_face,
            actions=["emotion"],
            enforce_detection=False
        )
        return analysis[0]["dominant_emotion"]

    except Exception:
        return ""


# ==========================================
# SHARED STATE (protected by lock)
# ==========================================

state_lock = threading.Lock()

current_name = "Unknown"
current_emotion = ""
current_box = None
latest_reply = "Hello!"

last_name = ""
last_emotion = ""
last_chat_time = 0
chat_delay = 5


# ==========================================
# BACKGROUND OLLAMA THREAD
# ==========================================

reply_queue = queue.Queue()


def chatbot_worker():

    global latest_reply

    while True:

        item = reply_queue.get()

        if item is None:
            break

        name, emotion = item

        reply = ask_ollama(name, emotion)

        with state_lock:
            latest_reply = reply

        reply_queue.task_done()


threading.Thread(target=chatbot_worker, daemon=True).start()


# ==========================================
# BACKGROUND RECOGNITION + EMOTION THREAD
# This is what removes DeepFace from the main camera loop entirely.
# maxsize=1 means: if the worker is busy, new frames are just dropped
# instead of piling up (keeps things real-time instead of laggy).
# ==========================================

vision_queue = queue.Queue(maxsize=1)


def vision_worker():

    global current_name, current_emotion, last_name, last_emotion, last_chat_time

    while True:

        item = vision_queue.get()

        if item is None:
            break

        frame, box = item
        x, y, w, h = box

        pad_x, pad_y = int(w * 0.4), int(h * 0.4)
        x1 = max(0, x - pad_x)
        y1 = max(0, y - pad_y)
        x2 = min(frame.shape[1], x + w + pad_x)
        y2 = min(frame.shape[0], y + h + pad_y)
        padded_face = frame[y1:y2, x1:x2]

        name = recognize_face(frame, box)
        emotion = analyze_emotion(padded_face)

        with state_lock:
            current_name = name
            current_emotion = emotion

            changed = (name != last_name or emotion != last_emotion)

            if changed and (time.time() - last_chat_time > chat_delay):
                reply_queue.put((name, emotion))
                last_name = name
                last_emotion = emotion
                last_chat_time = time.time()

        vision_queue.task_done()


threading.Thread(target=vision_worker, daemon=True).start()


# ==========================================
# CAMERA
# ==========================================

cap = cv2.VideoCapture(0)

face_cascade = cv2.CascadeClassifier(
    cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
)

frame_counter = 0
DETECT_EVERY_N_FRAMES = 5  # Haar is cheap, but no need to run every single frame

DETECTION_WIDTH = 640  # downscale for faster Haar detection

# ==========================================
# MAIN LOOP
# ==========================================

while True:

    ret, frame = cap.read()

    if not ret:
        break

    frame_counter += 1

    if frame_counter % DETECT_EVERY_N_FRAMES == 0:

        # Downscale just for detection speed, then map box back to full res
        h0, w0 = frame.shape[:2]
        scale = DETECTION_WIDTH / w0
        small = cv2.resize(frame, (DETECTION_WIDTH, int(h0 * scale)))
        gray = cv2.cvtColor(small, cv2.COLOR_BGR2GRAY)

        faces = face_cascade.detectMultiScale(
            gray,
            scaleFactor=1.3,
            minNeighbors=5
        )

        if len(faces) > 0:

            # Pick only the single largest (most prominent) face
            fx, fy, fw, fh = max(faces, key=lambda f: f[2] * f[3])

            # Map back to original frame coordinates
            x = int(fx / scale)
            y = int(fy / scale)
            w = int(fw / scale)
            h = int(fh / scale)

            with state_lock:
                current_box = (x, y, w, h)

            # Hand off to background worker; drop if it's still busy
            # with the previous frame instead of blocking the camera loop
            try:
                vision_queue.put_nowait((frame.copy(), (x, y, w, h)))
            except queue.Full:
                pass

        else:
            with state_lock:
                current_box = None

    # ----------------------------------
    # Drawing (uses whatever the background worker last computed)
    # ----------------------------------

    with state_lock:
        box = current_box
        name = current_name
        emotion = current_emotion
        reply = latest_reply

    if box is not None:

        x, y, w, h = box

        cv2.rectangle(frame, (x, y), (x + w, y + h), (0, 255, 0), 2)

        cv2.putText(
            frame, f"Name: {name}", (x, y - 40),
            cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2
        )

        cv2.putText(
            frame, f"Emotion: {emotion}", (x, y - 15),
            cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 0), 2
        )

        cv2.putText(
            frame, reply, (x, y + h + 25),
            cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 255), 2
        )

    cv2.imshow("Emotion Chatbot v2", frame)

    key = cv2.waitKey(1)

    if key == ord("q") or key == 27:
        break

# ==========================================
# CLEANUP
# ==========================================

reply_queue.put(None)
vision_queue.put(None)

cap.release()
cv2.destroyAllWindows()

Loading known faces...
Loaded: DongAn.jpg
Loaded: Harsh.jpg
Loaded: Sandra.jpg
Faces loaded: 3
